# Frontier-AI-lab competition — model notebook (draft v2)

*Companion widget: `app.py`. This notebook is the **single source of truth** for all model code:
every model function lives in a code cell whose first line is `#| export`. `notebook_loader.py`
concatenates those cells and runs them, so the Streamlit widget contains no model math. The
non-exported cells illustrate each function.*

The model asks two questions about a frontier **leader** $L$ (the frontier labs, collapsed into one
player) facing an open-weight **follower** $F$ (competitive fringe selling inference at cost):

- **(a) Profitability** — does the leader's profit flow $\Pi^L_t$ turn positive within ~5 years and
  stay positive under a compute slowdown?
- **(b) Release delay** — does withholding the frontier model (a delay $\tau$) pay, by slowing the
  follower's distillation?

**Units (spec N1).** States $a,c,x$ are in **OOMs** (base-10 logs of algorithmic level, training
compute, and effective compute $x=a+c$). State growth rates ($g_{C}$, $g_a$) are in **OOM/yr**;
diffusion rates $\delta$, discount $r$, price decline $g_p$, slowdown $\xi$ are **continuous per
year**; value curvature $\nu$ is **per OOM**. $t=0$ is early 2026. Normalization $x^L_0=0$: all
capabilities are OOMs relative to today's frontier. Dollars are anchored by two flow scales,
$W_0$ (value at today's frontier) and $S_0$ (today's frontier training spend), so absolute FLOP
counts never enter.

Structure mirrors the spec: **I.0** setup → **I.1** A (frontier) → **I.2** B (follower) →
**I.3** D (value) → **I.4** C (cost/profit) → **I.5** outputs → **II** extensions.

## I.0 — Setup and parameters

All parameters travel in one `Params` dataclass (defaults are the widget's provisional starting
point; **every default marked `provisional -> calibration session` is a placeholder** pending the
calibration session — the widget exposes each as a control). Capability is effective compute
$x^i_t = a^i_t + c^i_t$; the gap is $\Delta_t = x^L_t - x^F_t$. The follower starts a gap $\Delta_0$
behind, split into an algorithmic part (`split`·$\Delta_0$) and a compute part.

In [ ]:
#| export
# ---- Cell E1: imports + Params dataclass ----
import numpy as np
from dataclasses import dataclass, field, replace

@dataclass
class Params:
    # ----- Scenario -----
    T: float = 10.0            # horizon (yr), D-007
    dt: float = 0.005          # integrator step (yr), design N1/N6
    tau: float = 0.0           # release delay (yr); x^R_t = x^L_{t-tau}

    # ----- A: frontier progress -----
    A1: bool = False           # benchmark A1: adot_L == g_a (pure exogenous)
    g_C0: float = 0.623        # provisional -> calibration session  (=log10 4.2, OOM/yr)
    g_C_inf: float = 0.13      # provisional -> calibration session  (compute-growth floor, OOM/yr)
    xi: float = 0.3            # provisional -> calibration session  (slowdown decay, /yr)
    g_a: float = 0.447         # provisional -> calibration session  (=log10 2.8, OOM/yr)
    alpha: float = 0.5         # provisional -> calibration session  (experiment-compute weight)
    eta: float = 1.0           # base CES exponent, D-018 (1 => weighted average)
    leontief: bool = False     # eta -> -inf (min) option
    rho0: float = 0.3          # provisional -> calibration session  (today's AI R&D speedup)
    gamma: float = 0.2         # provisional -> calibration session  (psi compounding slope, /OOM; >~0.4 goes super-exponential inside the horizon, see N4)

    # ----- B: follower catch-up -----
    Delta0: float = 0.7        # provisional -> calibration session  (initial gap, OOM)
    split: float = 0.5         # provisional -> calibration session  (algo share of Delta0)
    g_a_F: float = 0.31        # provisional -> calibration session  (follower own algo rate, OOM/yr)
    delta_dev: float = 0.20    # stationary-lag calibration 2026-07-19 (developed-model diffusion, /yr)
    delta_rel: float = 0.26    # stationary-lag calibration 2026-07-19 (released-model distillation, /yr)
    g_CF0: float = 0.5         # provisional -> calibration session  (follower compute growth today, OOM/yr)
    g_CF_inf: float = 0.10     # provisional -> calibration session  (follower compute floor, OOM/yr)
    xi_F: float = 0.3          # provisional -> calibration session  (follower slowdown decay, /yr)

    # ----- D: value of capability -----
    W0: float = 350.0          # provisional -> calibration session  (W at today's frontier, $B/yr)
    nu: float = 0.55 / 2.302585092994046   # D-039: value-OOMs per capability-OOM (log10); observable 10^nu ~= 1.733x/OOM unchanged (was 0.55 natural)
    x_mid: float = 10.0        # provisional -> calibration session  (logistic bend location, OOM)

    # ----- C: conduct & cost -----
    theta: float = 0.35        # provisional -> calibration session  (conduct/operating-margin)
    phi_RD: float = 1.0        # provisional -> calibration session  (R&D markup on compute bill)
    ell: float = 0.5           # provisional -> calibration session  (build lag, yr)
    S0: float = 40.0           # provisional -> calibration session  (today's frontier train-compute spend, $B/yr)
    g_p: float = 0.56 / 2.302585092994046  # D-039: price decline in OOM/yr (log10); =log10(4.2/2.4) so cost growth matches Cottier's observed 2.4x/yr given g_C0. Hardware-only price-performance is ~1.35x/yr (g_p~0.13) -- the gap is a documented calibration tension
    r: float = 0.08            # provisional -> calibration session  (discount rate, /yr)

    # ----- Extensions (dials; default OFF) -----
    phi_mix: float = 0.0       # II.2 public-knowledge mix (effective a~_L blend)
    theta_gap: bool = False    # II.4 gap-dependent conduct (provisional form, D-027)
    theta_Delta_scale: float = 0.7   # II.4 scale Delta_theta (OOM)
    chi: float = 0.0           # II.5 distillation-defense dial (0..0.35)
    own_mode: bool = False     # II.6 ownership / user-cost variant (provisional)
    delta_K: float = 0.3       # II.6 GPU economic depreciation (/yr)
    labor_line: bool = False   # II.7 decoupled labor line (run only if headline positive)
    L0: float = 10.0           # II.7 labor level today ($B/yr)
    g_w: float = 0.05          # II.7 wage growth (/yr)

In [ ]:
# Illustration: the default parameter set. Note x_L0 = 0 (today's frontier is the origin).
p0 = Params()
print("Horizon T =", p0.T, "yr;  integrator step dt =", p0.dt, "yr")
print("Frontier compute growth today g_C0 =", p0.g_C0, "OOM/yr  (= log10 4.2x/yr)")
print("Algo progress today g_a =", p0.g_a, "OOM/yr  (= log10 2.8x/yr)")
print("Initial gap Delta0 =", p0.Delta0, "OOM;  algo share =", p0.split)
print("Value at today's frontier W0 =", p0.W0, "$B/yr;  curvature nu =", p0.nu, "/OOM")
print("Today's train spend S0 =", p0.S0, "$B/yr;  conduct theta =", p0.theta)

In [ ]:
# --- plotting style (dataviz palette; light surface). Not exported; illustration only. ---
import matplotlib.pyplot as plt
import matplotlib as mpl

PAL = {"blue": "#2a78d6", "green": "#008300", "magenta": "#e87ba4", "yellow": "#eda100",
       "aqua": "#1baf7a", "orange": "#eb6834", "violet": "#4a3aa7", "red": "#e34948",
       "good": "#0ca30c", "critical": "#d03b3b"}
INK, INK2, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.titlecolor": INK,
    "font.size": 11, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2.0,
    "figure.dpi": 110,
})

def style(ax, title=None, xlabel=None, ylabel=None):
    if title:  ax.set_title(title, fontsize=12, loc="left", pad=8)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.axhline(0, color=AXIS, lw=1, zorder=0)
    return ax
print("style ready")

## I.1 — Component A: frontier progress

**Compute** is exogenous with a decaying growth rate — the paper's driving experiment is a compute
slowdown, and $\xi$ (how fast today's rate $g_{C,0}$ decays toward the floor $g_{C,\infty}$) is a
headline scenario dial:
$$\dot c^L_t = g_{C}(t) = g_{C,\infty} + (g_{C,0}-g_{C,\infty})\,e^{-\xi t}\quad[\text{OOM/yr}].$$

**Algorithmic progress** is a CES aggregate of two inputs, each normalized to 1 at $t=0$ so that
$g_a$ is *today's* rate: researcher effort accelerated by AI assistance $\psi(x^L)$ (a mild RSI
feedback), and experiment throughput moving with compute growth:
$$\dot a^L_t = g_a\Big[(1-\alpha)\big(\tfrac{\psi(x^L_t)}{\psi(x^L_0)}\big)^{\eta}
              + \alpha\big(\tfrac{g_C(t)}{g_{C,0}}\big)^{\eta}\Big]^{1/\eta},
\qquad \psi(x)=1+\rho_0 e^{\gamma x}.$$
The exponent $\eta$ sets substitution: $\eta\!\to\!1$ (near-substitutes, weighted **average** —
a compute slowdown barely dents algo progress), $\eta\!\to\!0$ (Cobb–Douglas), $\eta<0$
(complements), $\eta\!\to\!-\infty$ (Leontief `min`, the scarcer input rules). **Base $\eta=1$**
(D-018). Benchmark **A1** sets $\dot a^L\equiv g_a$ (pure exogenous).

**Caveat (N4).** With $\gamma>0$ the $\psi$-feedback makes the law formally *super-exponential*.
The `psi_boost_share` diagnostic reports the fraction of $\dot a^L$ coming from $\psi$ having grown
above its initial value; the widget flags when it stops being small (>25%). **At the provisional
default $\gamma=0.5$ this feedback is NOT mild — it drives a finite-time singularity around
$t\approx 8$ yr** (v1's explosive regime). Set $\gamma=0$ ("freeze AI assistance") to see the
intended bounded harvest-vs-commoditization economics; the calibration session must revisit $\gamma$.

In [ ]:
#| export
# ---- Cell E2: Component A -- compute growth, psi, algo growth ----
def compute_growth(t, p):
    """g_C(t): frontier physical-compute growth (OOM/yr). Spec I.1.
    Exogenous decaying rate: g_C_inf + (g_C0 - g_C_inf) e^{-xi t}."""
    return p.g_C_inf + (p.g_C0 - p.g_C_inf) * np.exp(-p.xi * t)

def psi(x, p):
    """AI-assistance multiplier psi(x) = 1 + rho0 e^{gamma (x - x_L0)}, x_L0=0. Spec I.1.
    Exponent clipped for numerical safety in the explosive (super-exponential) regime; W-saturation
    then bounds revenue downstream. The clip only bites once psi is astronomically large (flagged)."""
    return 1.0 + p.rho0 * np.exp(np.clip(p.gamma * x, -700.0, 80.0))

def _ces_bracket(R_psi, R_gc, p):
    """CES aggregate of the two normalized inputs; handles eta=1, eta->0, eta<0, Leontief."""
    a = 1.0 - p.alpha
    b = p.alpha
    if p.leontief:                       # eta -> -inf : Leontief (scarcer input rules)
        return np.minimum(R_psi, R_gc)
    if abs(p.eta) < 1e-8:                 # eta -> 0 : Cobb-Douglas limit
        return R_psi**a * R_gc**b
    if abs(p.eta - 1.0) < 1e-12:          # eta = 1 : weighted average
        return a * R_psi + b * R_gc
    # general CES, guarded against overflow/NaN
    inner = a * R_psi**p.eta + b * R_gc**p.eta
    inner = np.maximum(inner, 1e-300)
    return inner**(1.0 / p.eta)

def algo_growth_L(t, x_L, p):
    """adot_L(t): leader algorithmic progress (OOM/yr). Spec I.1.
    g_a * [ (1-a)(psi(x_L)/psi(0))^eta + a (g_C(t)/g_C0)^eta ]^{1/eta}.
    Benchmark A1: adot_L == g_a."""
    if p.A1:
        return p.g_a
    R_psi = psi(x_L, p) / psi(0.0, p)          # research input, normalized to 1 at t=0
    R_gc = compute_growth(t, p) / p.g_C0       # experiment-compute input, normalized to 1 at t=0
    return p.g_a * _ces_bracket(R_psi, R_gc, p)

def psi_boost_share(t, x_L, p):
    """Fractional boost to adot_L coming from psi having grown above psi(0) (RSI feedback).
    0 at t=0; N4 says flag when it stops being small (>~25%)."""
    if p.A1:
        return np.zeros_like(np.asarray(t, dtype=float))
    R_psi = psi(x_L, p) / psi(0.0, p)
    R_gc = compute_growth(t, p) / p.g_C0
    full = _ces_bracket(R_psi, R_gc, p)
    frozen = _ces_bracket(np.ones_like(np.asarray(R_gc, dtype=float)) if np.ndim(R_gc) else 1.0, R_gc, p)
    full = np.asarray(full, dtype=float)
    frozen = np.asarray(frozen, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        share = 1.0 - frozen / full
    return np.where(full > 0, share, 0.0)

In [ ]:
# Illustration: compute-growth slowdown, the psi feedback, and the CES algo law across eta.
tt = np.linspace(0, 10, 300)
fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.8))

axs[0].plot(tt, [compute_growth(t, p0) for t in tt], color=PAL["blue"])
axs[0].axhline(p0.g_C_inf, color=AXIS, ls="--", lw=1)
axs[0].text(6, p0.g_C_inf+0.01, "floor $g_{C,\\infty}$", color=INK2, fontsize=9)
style(axs[0], "Compute growth $g_C(t)$", "year", "OOM/yr")

xx = np.linspace(0, 6, 200)
axs[1].plot(xx, psi(xx, p0)/psi(0.0, p0), color=PAL["orange"])
style(axs[1], "AI-assistance $\\psi(x)/\\psi(0)$  ($\\gamma$=0.5)", "capability $x$ (OOM)", "multiplier")

for name, kw, col in [("$\\eta$=1 (avg)", dict(eta=1.0), PAL["blue"]),
                      ("$\\eta$=0.61", dict(eta=0.61), PAL["green"]),
                      ("$\\eta$=0 (CD)", dict(eta=1e-9), PAL["magenta"]),
                      ("$\\eta$=-2", dict(eta=-2.0), PAL["orange"]),
                      ("min", dict(leontief=True), PAL["violet"]),
                      ("A1", dict(A1=True), AXIS)]:
    from dataclasses import replace as _rep
    pk = _rep(p0, gamma=0.0, **kw)   # gamma=0 here so we isolate the eta comparison
    axs[2].plot(tt, [algo_growth_L(t, 0.4*t, pk) for t in tt], color=col, label=name,
                lw=2 if name != "A1" else 1.4, ls="-" if name != "A1" else "--")
style(axs[2], "Algo law $\\dot a^L(t)$ by $\\eta$  ($\\gamma$=0)", "year", "OOM/yr")
axs[2].legend(frameon=False, fontsize=8, loc="upper right")
fig.tight_layout(); plt.show()

# psi-share diagnostic (share of algo progress coming from RSI feedback) vs capability, gamma=0.5.
# Rises fast -> the widget flags >25%. (Shown vs x here; simulate() traces it along the path later.)
xg = np.linspace(0, 6, 120)
print("psi_boost_share at x=[0,2,4,6] (t=3):",
      [round(float(psi_boost_share(3.0, x, p0)), 3) for x in [0,2,4,6]])

## I.2 — Component B: follower catch-up

What catches up is the follower's *algorithmic level* — compute cannot be copied:
$$\dot a^F_t = g^F_a
 + \underbrace{\delta_{dev}\,(a^L_t-a^F_t)}_{\text{developed-model: espionage, papers, mobility}}
 + \underbrace{\delta_{rel}\,\max(x^R_t-x^F_t,\,0)}_{\text{released-model: distillation}}.$$
The **developed channel** runs on the algorithmic gap (methods diffuse). The **released channel**
runs on the gap to the *served* capability $x^R$ — distillation copies what the served model can do,
so it is the leader-controlled lever for question (b). Both cross terms measure imitation only and
die at parity (N2, D-015); the $\max(\cdot,0)$ enforces the cap $x^F\le x^R$. Follower compute
$c^F$ is exogenous (same decaying family, its own parameters). The observed open–closed lag pins
the **total** $\delta_{dev}+\delta_{rel}$; the *split* is the free parameter question (b) turns on.

In [ ]:
#| export
# ---- Cell E3: Component B -- follower compute growth and algo catch-up ----
def follower_compute_growth(t, p):
    """g_CF(t): follower physical-compute growth (OOM/yr). Same decaying family, follower params."""
    return p.g_CF_inf + (p.g_CF0 - p.g_CF_inf) * np.exp(-p.xi_F * t)

def algo_growth_F(aL, aF, xR, xF, p):
    """adot_F: follower algorithmic progress (OOM/yr). Spec I.2.
    g_a_F + delta_dev (a~L - aF) + (1-chi) delta_rel * max(xR - xF, 0).
    II.2: a~L = (1-phi_mix) aL + phi_mix aF blends in the public stock.
    II.5: chi throttles the released-model (distillation) channel."""
    aL_eff = (1.0 - p.phi_mix) * aL + p.phi_mix * aF          # II.2 (phi_mix=0 -> aL)
    dev = p.delta_dev * (aL_eff - aF)                          # developed-model channel
    rel = (1.0 - p.chi) * p.delta_rel * np.maximum(xR - xF, 0.0)  # released-model channel (cap x^F<=x^R)
    return p.g_a_F + dev + rel

In [ ]:
# Illustration: follower compute-growth path, and the two catch-up channels vs the gaps they run on.
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.8))
axs[0].plot(tt, [follower_compute_growth(t, p0) for t in tt], color=PAL["green"])
axs[0].plot(tt, [compute_growth(t, p0) for t in tt], color=PAL["blue"], ls="--", lw=1.5)
axs[0].text(6, follower_compute_growth(6,p0)+0.01, "follower", color=PAL["green"], fontsize=9)
axs[0].text(6, compute_growth(6,p0)+0.01, "leader", color=PAL["blue"], fontsize=9)
style(axs[0], "Compute growth: leader vs follower", "year", "OOM/yr")

gaps = np.linspace(0, 1.5, 100)
axs[1].plot(gaps, p0.delta_dev*gaps, color=PAL["magenta"], label="developed  $\\delta_{dev}\\,\\Delta a$")
axs[1].plot(gaps, p0.delta_rel*gaps, color=PAL["orange"], label="released  $\\delta_{rel}\\,\\Delta x^R$")
axs[1].plot(gaps, (p0.delta_dev+p0.delta_rel)*gaps, color=INK2, ls="--", lw=1.4, label="total $\\delta$")
style(axs[1], "Catch-up contribution vs gap", "gap (OOM)", "OOM/yr added to $\\dot a^F$")
axs[1].legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout(); plt.show()

## I.3 — Component D: value of capability $W(x)$

$W(x)$ is the flow of dollars capability $x$ commands — a single logistic family, anchored so that
$W(0)=W_0$ exactly for any $(\nu,x_{mid})$:
$$W(x)=\frac{W_{max}}{1+e^{-\nu(x-x_{mid})}},\qquad W(0)\equiv W_0.$$
**Curvature is the pivotal assumption.** For $x\ll x_{mid}$ (bend beyond the horizon) the logistic is
the exponential $W\approx w_0 e^{\nu x}$: a defended gap earns *growing* rents (**harvest**). Bring
the bend $x_{mid}$ inside the horizon and the same gap earns *shrinking* rents
(**commoditization**). So *where the bend sits* is the decision-relevant knob (N8). II.4 adds a
second commoditization route: conduct $\theta$ made increasing in the served gap (price-war
evidence), $\theta(\Delta)=\bar\theta\,(1-e^{-\Delta/\Delta_\theta})/(1-e^{-\Delta_0/\Delta_\theta})$
— provisional (D-027), normalized to $\bar\theta$ at today's gap.

In [ ]:
#| export
# ---- Cell E4: Component D -- value of capability, and II.4 conduct ----
def W(x, p):
    """W(x): flow of dollars capability x commands ($B/yr). Spec I.3.
    BASE-10 logistic (D-039: nu = value-OOMs per capability-OOM), anchored so W(0) = W0 exactly
    for any (nu, x_mid)."""
    x = np.asarray(x, dtype=float)
    nl = p.nu * np.log(10.0)                                  # log10 slope -> natural, for exp()
    W_max = p.W0 * (1.0 + np.exp(np.clip(nl * p.x_mid, -700.0, 700.0)))   # anchor: W(0)=W0
    z = np.clip(-nl * (x - p.x_mid), -700.0, 700.0)           # numerical safety; W saturates at W_max
    return W_max / (1.0 + np.exp(z))

def W_exp_approx(x, p):
    """Exponential-regime approximation w0 * 10^{nu x}, w0 = W_max * 10^{-nu x_mid}. Spec I.3."""
    x = np.asarray(x, dtype=float)
    nl = p.nu * np.log(10.0)
    W_max = p.W0 * (1.0 + np.exp(np.clip(nl * p.x_mid, -700.0, 700.0)))
    w0 = W_max * np.exp(np.clip(-nl * p.x_mid, -700.0, 700.0))
    return w0 * np.exp(np.clip(nl * x, -700.0, 700.0))

def theta_t(gap, p):
    """Conduct parameter. Constant theta when II.4 off; gap-dependent (provisional, D-027) when on.
    theta(Delta) = theta_bar (1-e^{-Delta/Ds}) / (1-e^{-Delta0/Ds}), normalized to theta_bar at Delta0."""
    if not p.theta_gap:
        return p.theta
    Ds = p.theta_Delta_scale
    gap = np.asarray(gap, dtype=float)
    num = 1.0 - np.exp(-np.maximum(gap, 0.0) / Ds)
    den = 1.0 - np.exp(-p.Delta0 / Ds)
    return p.theta * num / den

In [ ]:
# Illustration: logistic W vs exponential approximation, and the II.4 conduct dial.
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.8))
xx = np.linspace(0, 6, 200)
axs[0].plot(xx, W(xx, p0), color=PAL["blue"], label="logistic $W(x)$")
axs[0].plot(xx, W_exp_approx(xx, p0), color=PAL["orange"], ls="--", lw=1.6, label="exp. approx $w_0e^{\\nu x}$")
axs[0].scatter([0], [p0.W0], color=PAL["blue"], zorder=5, s=28)
axs[0].text(0.1, p0.W0*1.15, "$W(0)=W_0$", color=INK2, fontsize=9)
style(axs[0], "Value map $W(x)$  ($x_{mid}$=10, $\\nu$=0.55)", "capability $x$ (OOM)", "$/yr value  ($B)")
axs[0].legend(frameon=False, fontsize=9, loc="upper left")

from dataclasses import replace as _rep
pth = _rep(p0, theta_gap=True)
gg = np.linspace(0, 1.6, 100)
axs[1].plot(gg, [float(theta_t(g, pth)) for g in gg], color=PAL["violet"])
axs[1].axhline(p0.theta, color=AXIS, ls="--", lw=1)
axs[1].axvline(p0.Delta0, color=AXIS, ls=":", lw=1)
axs[1].text(p0.Delta0+0.03, 0.05, "today's gap $\\Delta_0$", color=INK2, fontsize=9, rotation=90, va="bottom")
style(axs[1], "II.4 gap-dependent conduct $\\theta(\\Delta)$", "served gap (OOM)", "$\\theta$")
fig.tight_layout(); plt.show()

## I.4 — Component C: cost and profit

$$\Pi^L_t = \theta\,[\,W(x^R_t)-W(x^F_t)\,]\;-\;(1+\varphi_{RD})\,p_t\,C^L_{t+\ell},
\qquad p_t=p_0\, 10^{-g_p t}.$$
Revenue runs on the **served** capability $x^R$ (the leader can only monetize what it serves).
$\theta$ is a single conduct parameter — the product $\theta\times$rent is *operating* profit
(competition, imperfect extraction, and serving costs all inside $\theta$). Compute is rented and
**paid when used** (D-021): at $t$ the leader pays today's price for the compute of the model that
arrives — and starts earning — a build lag $\ell$ later; the NPV integral's discounting does the
rest. In flow form, anchored to today's spend $S_0$ (spec N3):
$$\text{cost}_t=(1+\varphi_{RD})\,S_0\,10^{\,c^L(t+\ell)-c^L(0)}\,10^{-g_p t}.$$
The $10^{\Delta c}$ factor tracks compute growth; $10^{-g_p t}$ is hardware price-performance only (D-039: $g_p$ in OOM/yr)
(algorithmic progress is already in $x$ — don't double-count). At $t=0$ this is $(1+\varphi_{RD})S_0$.

In [ ]:
#| export
# ---- Cell E5: Component C -- cost flow ----
def cost_flow(t, c_L_at, c_L_ell, p):
    """Leader training-cost flow ($B/yr). Spec I.4 / N3.
    cost = (1+phi_RD) S0 * 10^{c_L(t+ell) - c_L(0)} * 10^{-g_p t}, and c_L(0)=0 in the OOM convention,
    so this is (1+phi_RD) S0 * 10^{c_L(t+ell)} * 10^{-g_p t}.
    c_L_at = c_L(t+ell). (c_L_ell is retained in the signature for the caller but NO LONGER
    subtracted -- ANCHOR CHANGED to c_L(0), D-036 4th amendment: ell now has a level effect BY DESIGN
    -- the firm pays TODAY for the compute of the model shipping at t+ell, so a longer lead time ell
    raises today's bill by 10^{c_L(ell)}. S0 = compute bill of the CURRENT frontier model today.)

    CALIBRATION NOTE: at full defaults (phi_RD=1, ell=0.5) this makes cost(0) ~= 164 $B/yr (was ~80
    under the old c_L(ell) anchor). S0 is NOT re-fit here -- flagged as a calibration-session item.
    The BASE form (phi_RD=0) is cost = S0 * 10^{c_L(t+ell)} * 10^{-g_p t}. phi_RD>0 ADDS R&D overhead."""
    base = (1.0 + p.phi_RD) * p.S0 * 10.0**(c_L_at - p.g_p * t)   # D-039: g_p in OOM/yr
    if p.own_mode:
        # II.6 (provisional): Jorgenson user cost u = p_K (r + delta_K + g_p); competitive-rental
        # equivalent has price r + g_p, so owners additionally bear delta_K/(r+g_p). SIMPLIFIED:
        # captures the depreciation load, NOT the full CAPEX front-loading (flagged provisional).
        # r is a natural rate, so g_p (log10, D-039) is converted to natural units here.
        gp_nat = p.g_p * np.log(10.0)
        base = base * (p.r + p.delta_K + gp_nat) / (p.r + gp_nat)
    return base

In [ ]:
# Illustration: the cost flow and its two forces (compute growth up, price decline down).
# c_L(t) is the closed-form integral of compute_growth: g_Cinf*t + (g_C0-g_Cinf)(1-e^{-xi t})/xi.
def _cL_closed(t, p):
    return p.g_C_inf*t + (p.g_C0 - p.g_C_inf)*(1 - np.exp(-p.xi*t))/p.xi
_grid = np.arange(0.0, p0.T + 1.5 + p0.dt/2, p0.dt)
cL_ell = float(_cL_closed(p0.ell, p0))
cL_at = _cL_closed(_grid + p0.ell, p0)
cost = cost_flow(_grid, cL_at, cL_ell, p0)
m = _grid <= p0.T + 1e-9
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.plot(_grid[m], cost[m], color=PAL["critical"])
ax.plot(_grid[m], (1+p0.phi_RD)*p0.S0*10**(cL_at[m]-cL_ell), color=PAL["orange"], ls="--", lw=1.5,
        label="without price decline")
ax.text(0.2, (1+p0.phi_RD)*p0.S0*1.05, "$t$=0: $(1{+}\\varphi_{RD})S_0$ = %.0f $B/yr" % ((1+p0.phi_RD)*p0.S0),
        color=INK2, fontsize=9)
style(ax, "Leader training-cost flow", "year", "$/yr cost  ($B)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
fig.tight_layout(); plt.show()

## Numerical integration and the full system (N1/N6)

Fixed-step **RK4** on a $dt=0.005$ yr grid (a fixed grid is needed for the delayed term). The
**leader is autonomous** — its dynamics depend on neither the follower nor $x^R$ — so it is
integrated first on a half-step grid; the follower is then integrated on the coarse grid, reading
the *delayed* leader path $x^R(t)=x^L(t-\tau)$ off the fine grid by index (exact, no interpolation).
For $t<\tau$ the pre-period extrapolation $x^L(0)+(t-\tau)(g_{C,0}+g_a)$ is used. The state is
integrated to $T+1.5$ yr so `cost_flow` can read $c^L(t+\ell)$; outputs are truncated to $[0,T]$.
`simulate` returns the full time series plus diagnostics (`psi_share`, the two N2 cap booleans).

In [ ]:
#| export
# ---- Cell E6: RK4 integrator + full simulation ----
# The leader (a_L, c_L) is autonomous -- its dynamics depend on neither the follower nor x^R -- so
# it is integrated FIRST, on a half-step "fine" grid. The follower is then integrated on the coarse
# grid; each RK4 stage falls exactly on a fine-grid node, so the delayed leader path x^R(t)=x_L(t-tau)
# is read off by direct indexing (no interpolation) -- exact and fast.
def _rk4_leader_fine(fine, p):
    """Integrate the leader (a_L, c_L) on the fine grid (step dt/2) with fixed-step RK4."""
    n = len(fine)
    aL = np.zeros(n); cL = np.zeros(n)
    aL[0] = 0.0; cL[0] = 0.0                     # normalization x_L0 = 0
    for i in range(n - 1):
        h = fine[i+1] - fine[i]
        t = fine[i]; a = aL[i]; c = cL[i]
        k1a = algo_growth_L(t, a + c, p);             k1c = compute_growth(t, p)
        k2a = algo_growth_L(t+h/2, a+h/2*k1a + c+h/2*k1c, p); k2c = compute_growth(t+h/2, p)
        k3a = algo_growth_L(t+h/2, a+h/2*k2a + c+h/2*k2c, p); k3c = compute_growth(t+h/2, p)
        k4a = algo_growth_L(t+h,   a+h*k3a   + c+h*k3c,   p); k4c = compute_growth(t+h, p)
        aL[i+1] = a + h/6*(k1a + 2*k2a + 2*k3a + k4a)
        cL[i+1] = c + h/6*(k1c + 2*k2c + 2*k3c + k4c)
    return aL, cL

def _xR_of(s, grid, xL, p):
    """Released capability x^R at time argument s = t - tau. History lookup on the grid, with the
    design's pre-period backward extrapolation x_L(0) + (t-tau)(g_C0 + g_a) for s < 0."""
    s = np.asarray(s, dtype=float)
    pre = 0.0 + s * (p.g_C0 + p.g_a)             # x_L0 = 0; pre-2026 trend ~ today's rates
    interp = np.interp(np.clip(s, grid[0], grid[-1]), grid, xL)
    return np.where(s < 0.0, pre, interp)

def _rk4_follower_fine(coarse, aL_fine, xR_fine, gCF_fine, p):
    """Integrate the follower (a_F, c_F) on the coarse grid. Stage inputs (leader algo aL, released
    capability xR, follower compute growth gCF) are read by index from the fine grid: coarse node i
    is fine node 2i; the half-step stages use fine node 2i+1; the full step uses 2i+2."""
    n = len(coarse)
    aF = np.zeros(n); cF = np.zeros(n)
    aF[0] = 0.0 - p.split * p.Delta0             # a_F0 : algo part of the initial gap
    cF[0] = 0.0 - (1.0 - p.split) * p.Delta0     # c_F0 : compute part of the initial gap
    for i in range(n - 1):
        h = coarse[i+1] - coarse[i]
        j = 2*i
        a = aF[i]; c = cF[i]
        k1a = algo_growth_F(aL_fine[j],   a,          xR_fine[j],   a + c,          p); k1c = gCF_fine[j]
        k2a = algo_growth_F(aL_fine[j+1], a+h/2*k1a, xR_fine[j+1], a+h/2*k1a + c+h/2*k1c, p); k2c = gCF_fine[j+1]
        k3a = algo_growth_F(aL_fine[j+1], a+h/2*k2a, xR_fine[j+1], a+h/2*k2a + c+h/2*k2c, p); k3c = gCF_fine[j+1]
        k4a = algo_growth_F(aL_fine[j+2], a+h*k3a,   xR_fine[j+2], a+h*k3a   + c+h*k3c,   p); k4c = gCF_fine[j+2]
        aF[i+1] = a + h/6*(k1a + 2*k2a + 2*k3a + k4a)
        cF[i+1] = c + h/6*(k1c + 2*k2c + 2*k3c + k4c)
    return aF, cF

def simulate(p, tau_fn=None):
    """Integrate the full system and return time series on [0, T]. Spec I.0-I.5.
    Internally integrates PAST the horizon so cost_flow can read c_L(t+ell) for every displayed
    t <= T; outputs truncated to T."""
    # The pad must COVER the training lead ell (slider allows up to 3 yr): with a fixed pad the
    # c_L(t+ell) interpolation silently clamped at the grid end, freezing the compute factor while
    # 10^{-g_p t} kept falling -> a spurious cost peak/decline kink at t = T - (ell - pad)
    # (Pavel's bug report 2026-07-23, horizon 5 / ell 3 -> kink at 3.5). 1.5 stays as the floor.
    pad = max(1.5, float(p.ell) + 2.0 * float(p.dt))
    grid = np.arange(0.0, p.T + pad + p.dt/2, p.dt)          # coarse grid, step dt
    fine = np.arange(0.0, p.T + pad + p.dt/4, p.dt/2)        # fine grid, step dt/2
    fine = fine[:2*len(grid)-1]                              # align: fine node 2i == coarse node i
    aL_f, cL_f = _rk4_leader_fine(fine, p)
    xL_f = aL_f + cL_f
    # tau may be the constant p.tau, or a time-varying schedule tau_fn(t) passed as a callable
    # (e.g. a delay that switches on for a window): released path is x^R(t) = x_L(t - tau(t)).
    tau_arr = np.asarray(tau_fn(fine), dtype=float) if callable(tau_fn) else p.tau
    xR_f = _xR_of(fine - tau_arr, fine, xL_f, p)             # delayed leader path on the fine grid
    gCF_f = follower_compute_growth(fine, p)
    aF, cF = _rk4_follower_fine(grid, aL_f, xR_f, gCF_f, p)
    # sample leader arrays back onto the coarse grid
    aL = aL_f[::2]; cL = cL_f[::2]; xL = xL_f[::2]; xR = xR_f[::2]
    xF = aF + cF
    Delta = xL - xF

    # value flows on the served vs follower capability
    W_R = W(xR, p)
    W_F = W(xF, p)
    served_gap = xR - xF
    theta_series = theta_t(served_gap, p) if np.ndim(theta_t(served_gap, p)) else np.full_like(grid, theta_t(served_gap, p))
    theta_series = np.broadcast_to(theta_series, grid.shape).astype(float)
    rent = np.maximum(W_R - W_F, 0.0)            # N2: revenue floors at zero if W_R < W_F
    revenue = theta_series * rent
    revenue = revenue * (1.0 - 0.4 * p.chi)      # II.5: defense costs 0.4*chi of revenue

    # cost: c_L(t+ell) via interpolation; c_L(ell) constant
    cL_ell = float(np.interp(p.ell, grid, cL))
    cL_at = np.interp(grid + p.ell, grid, cL)    # t+ell within padded grid for t<=T
    cost = cost_flow(grid, cL_at, cL_ell, p)

    profit = revenue - cost
    if p.labor_line:                             # II.7 decoupled labor line
        profit = profit - p.L0 * np.exp(p.g_w * grid)

    # diagnostics
    psi_share = np.asarray(psi_boost_share(grid, xL, p), dtype=float)
    cap_xF_le_xR = xF <= xR + 1e-9               # slack good
    cap_W = W_R >= W_F - 1e-9

    # truncate to [0, T]
    m = grid <= p.T + 1e-9
    t = grid[m]
    disc = np.exp(-p.r * t)
    npv_integrand = disc * profit[m]
    npv_cum = np.concatenate([[0.0], np.cumsum((npv_integrand[1:] + npv_integrand[:-1]) / 2.0 * np.diff(t))])
    # undiscounted running total of profit (integral of Pi); the widget shows this, not NPV
    prof_m = profit[m]
    cum_profit = np.concatenate([[0.0], np.cumsum((prof_m[1:] + prof_m[:-1]) / 2.0 * np.diff(t))])

    out = dict(
        t=t, a_L=aL[m], c_L=cL[m], x_L=xL[m], a_F=aF[m], c_F=cF[m], x_F=xF[m], x_R=xR[m],
        Delta=Delta[m], W_R=W_R[m], W_F=W_F[m], theta=theta_series[m],
        revenue=revenue[m], cost=cost[m], profit=profit[m], npv_cum=npv_cum, cum_profit=cum_profit,
        psi_share=psi_share[m], cap_xF_le_xR=cap_xF_le_xR[m], cap_W=cap_W[m],
    )
    return out

In [ ]:
# Illustration: capability paths and the gap under the BOUNDED regime (gamma=0), so the economics
# are visible rather than the explosive default. x^R = x^L here (tau=0).
from dataclasses import replace as _rep
p_ill = _rep(p0, gamma=0.0)
s = simulate(p_ill)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].plot(s['t'], s['x_L'], color=PAL["blue"], label="leader $x^L$")
axs[0].plot(s['t'], s['x_F'], color=PAL["orange"], label="follower $x^F$")
axs[0].text(s['t'][-1], s['x_L'][-1], " $x^L$", color=PAL["blue"], fontsize=9, va="center")
axs[0].text(s['t'][-1], s['x_F'][-1], " $x^F$", color=PAL["orange"], fontsize=9, va="center")
style(axs[0], "Capability paths (bounded, $\\gamma$=0)", "year", "OOM above 2026 frontier")
axs[1].plot(s['t'], s['Delta'], color=PAL["violet"])
style(axs[1], "Capability gap $\\Delta_t = x^L-x^F$", "year", "OOM")
fig.tight_layout(); plt.show()
print("terminal gap:", round(float(s['Delta'][-1]), 3), "OOM;  caps ok:",
      bool(s['cap_xF_le_xR'].all() and s['cap_W'].all()))

## I.5 — Outputs: headline (a) and release-delay comparison (b)

**(a)** the path of $\Pi^L_t$: does it turn positive within ~5 yr and *stay* positive? NPV
$\int_0^T e^{-rt}\Pi^L_t\,dt$ is a secondary statistic. **(b)** compare $\Pi^L_t$ across release
delays $\tau$: withholding forgoes current revenue $\theta[W(x^L_t)-W(x^L_{t-\tau})]$ but slows the
follower through $\delta_{rel}$. `headline` also returns the implied $t=0$ operating profit
$\theta[W(x^R_0)-W(x^F_0)]$ for the cross-consistency check against observed ~\$40–60B/yr.

In [ ]:
#| export
# ---- Cell E7: headline statistics (question a) and delay comparison (question b) ----
def headline(sim, p):
    """Question (a) outputs. Spec I.5a.
    first sign crossing (yr), positive-within-5yr, stays-positive, NPV, terminal gap."""
    t = sim['t']; profit = sim['profit']
    pos = profit >= 0.0
    # first sign crossing: earliest t at which profit is non-negative
    idx = np.argmax(pos) if pos.any() else -1
    sign_crossing_year = float(t[idx]) if idx >= 0 else float('nan')
    within5 = bool(pos[t <= 5.0 + 1e-9].any())
    # stays positive from the crossing to T
    stays_positive = bool(idx >= 0 and pos[idx:].all())
    npv = float(sim['npv_cum'][-1])
    terminal_gap = float(sim['Delta'][-1])
    # cross-consistency: implied t=0 operating profit theta*[W(x_R0)-W(x_F0)]
    op_profit_t0 = float(sim['revenue'][0])
    return dict(
        sign_crossing_year=sign_crossing_year,
        positive_within_5yr=within5,
        stays_positive=stays_positive,
        npv=npv,
        terminal_gap=terminal_gap,
        op_profit_t0=op_profit_t0,
        cost_t0=float(sim['cost'][0]),
        psi_share_max=float(np.nanmax(sim['psi_share'])),
        cap_ok=bool(sim['cap_xF_le_xR'].all() and sim['cap_W'].all()),
    )

def delay_comparison(p, taus=(0.0, 0.25, 0.5, 1.0, 1.5, 2.0)):
    """Question (b). Spec I.5b. Pi_t paths and NPV as a function of release delay tau (yr)."""
    res = {'taus': list(taus), 'paths': [], 'npvs': [], 't': None}
    for tau in taus:
        s = simulate(replace(p, tau=tau))
        if res['t'] is None:
            res['t'] = s['t']
        res['paths'].append(s['profit'])
        res['npvs'].append(float(s['npv_cum'][-1]))
    res['npvs'] = np.array(res['npvs'])
    best_i = int(np.argmax(res['npvs']))
    res['best_tau'] = taus[best_i]
    res['best_npv'] = float(res['npvs'][best_i])
    return res

In [ ]:
# Illustration: headline at the bounded illustrative params, with the profit/revenue/cost picture.
h = headline(s, p_ill)
for k, v in h.items():
    print(f"  {k:22s} {v}")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].plot(s['t'], s['revenue'], color=PAL["green"], label="revenue $\\theta\\,\\Delta W$")
axs[0].plot(s['t'], s['cost'], color=PAL["critical"], label="cost")
style(axs[0], "Revenue vs cost", "year", "$/yr  ($B)")
axs[0].legend(frameon=False, fontsize=9, loc="upper left")
axs[1].plot(s['t'], s['profit'], color=PAL["blue"])
axs[1].fill_between(s['t'], s['profit'], 0, where=s['profit']>=0, color=PAL["good"], alpha=0.18)
axs[1].fill_between(s['t'], s['profit'], 0, where=s['profit']<0, color=PAL["critical"], alpha=0.14)
style(axs[1], "Profit flow $\\Pi^L_t$", "year", "$/yr  ($B)")
fig.tight_layout(); plt.show()

## Targets (D-037) — observables the widget and the Monte Carlo state in natural units

Targets-first parameterization: wherever a parameter has a clean observable, the *observable* is
the primitive — slider bounds, Monte-Carlo distributions and calibration documentation all live in
target space (`TARGET_RANGES`, one source of truth), and `invert_targets` maps them into model
parameters at $t=0$ (D-037 Q2). Defaults are the forward images of `Params()`, so the default
targets invert *exactly* back to the parameter defaults.

In [ ]:
#| export
# ---- Cell E9: targets (D-037) -- observable targets: ranges, forward map, inversions ----
# One source of truth per target: TARGET_RANGES drives the widget slider bounds, the Monte-Carlo
# distribution, and the calibration cards. Defaults are the FORWARD images of Params() defaults
# (stored exact, displayed rounded), so the default targets invert exactly back to Params().

TARGET_PARAM = {          # target key -> the parameter it pins (t_lag_mo also sets the deltas)
    't_compute_x': 'g_C0',    # compute scaling today, x/yr           g_C0 = log10(.)
    't_algo_x':    'g_a',     # algorithmic progress today, x/yr      g_a = log10(.)
    't_lag_mo':    'Delta0',  # follower lag, months                  Delta0 = lag_yr * leader speed
    't_bill_x':    'g_p',     # training-bill growth today, x/yr      g_p = g_C0 - log10(.)
    't_profit_B':  'W0',      # frontier operating profit today, $B/yr (W0 absorbs it, D-037 Q1)
    't_value_x':   'nu',      # value multiplier per OOM, x           nu = log10(.)
    't_floor_x':   'g_C_inf', # long-run compute-growth floor, x/yr   g_C_inf = log10(.)
}

def _profit_scale(base):
    """Operating profit today per unit of W0: theta * [W(0) - W(-Delta0)] at W0 = 1 (W is linear
    in W0). Uses base's theta, nu, Delta0 AND x_mid -- so the inversion is level-aware (the
    exponential-value levels pin x_mid huge, the logistic levels use the dial)."""
    pw = replace(base, W0=1.0)
    return float(base.theta * (W(0.0, pw) - W(-base.Delta0, pw)))

def target_defaults(p=None):
    """Forward map Params -> target values (exact; round-trips through invert_targets)."""
    p = p if p is not None else Params()
    return {
        't_compute_x': float(10.0**p.g_C0),
        't_algo_x':    float(10.0**p.g_a),
        't_lag_mo':    float(p.Delta0 / (p.g_C0 + p.g_a) * 12.0),
        't_bill_x':    float(10.0**(p.g_C0 - p.g_p)),
        't_profit_B':  float(p.W0 * _profit_scale(p)),
        't_value_x':   float(10.0**p.nu),
        't_floor_x':   float(10.0**p.g_C_inf),
    }

_TD0 = target_defaults()
# D-042 two-tier ranges: TARGET_RANGES is the ENVELOPE — the outer bounds of what a user may
# sample (the wide vetted priors of D-040). The DEFAULT simulation range is the TIGHT span of
# the documented sources (SIM_DEFAULT, cell E8b); dimensions without one default to a POINT and
# are not sampled until the user widens them. Endpoints that were images of round parameter-space
# bounds are now stated directly in natural units (same values up to that rounding).
TARGET_RANGES = {
    't_compute_x': ('uniform', 3.6, 5.3),                      # x/yr (was 10^[0.556, 0.724])
    't_algo_x':    ('uniform', 2.0, 3.5),                      # x/yr (was 10^[0.30, 0.544])
    't_lag_mo':    ('lognormal', float(np.log(np.sqrt(3.5 * 15.7))),
                    float(np.log(15.7 / 3.5) / 3.29)),         # ~90% CI [3.5, 15.7] mo; lower end
                                                               # widened 3.9 -> 3.5 so the Epoch ECI
                                                               # source stays inside the envelope
    't_bill_x':    ('uniform', 1.8, 3.2),                      # D-037: bill-growth stated directly
    't_profit_B':  ('uniform', 100.0 * _profit_scale(Params()),
                    1000.0 * _profit_scale(Params())),         # image of W0 in [100, 1000]
    't_value_x':   ('uniform', float(np.exp(0.1)), float(np.exp(1.1))),   # image of nu
    't_floor_x':   ('uniform', 10.0**0.05, 10.0**0.30),        # 1.12x .. 2.0x /yr (image of g_C_inf)
}

def channels_from_lag(lag_yr, speed, own_speed):
    """Wedge-split at the channels levels (L6+): rescale the jointly calibrated channel defaults
    (delta_dev, delta_rel) = (0.20, 0.26) so the stated lag is stationary in the same sense at the
    CURRENT speeds: kappa = (wedge/Delta0) / (wedge0/Delta0_0), with wedge = leader speed minus the
    follower's own speed (both at t=0) and Delta0 = lag_yr*speed. kappa = 1 at the default lag and
    speeds, so the notebook defaults are recovered exactly (D-037)."""
    p0 = Params()
    Delta0 = lag_yr * speed
    wedge = max(speed - own_speed, 0.0)
    wedge0 = (p0.g_C0 + p0.g_a) - (p0.g_a_F + p0.g_CF0)
    kappa = (wedge / max(Delta0, 1e-9)) / (wedge0 / p0.Delta0)
    return float(p0.delta_dev * kappa), float(p0.delta_rel * kappa)

def invert_targets(targets, base, merged=True):
    """Invert a dict of target values into a dict of model parameters (t=0 inversion, D-037 Q2).

    `base` supplies every parameter no target pins (theta, split, follower engine, x_mid, ...);
    inversions see already-inverted values (cross-target coupling by design: the lag inversion uses
    the compute/algo targets' speed; the profit inversion uses the inverted nu and Delta0).
    merged=True (pure-catch-up levels 1-5): the lag pins the single merged delta = 12/lag_mo, so
    delta*Delta0 = leader speed EXACTLY -- the gap is stationary for any lag position.
    merged=False (channels levels 6+): the lag pins Delta0 and rescales (delta_dev, delta_rel)
    via channels_from_lag."""
    out = {}
    if 't_compute_x' in targets:
        out['g_C0'] = float(np.log10(targets['t_compute_x']))
    if 't_algo_x' in targets:
        out['g_a'] = float(np.log10(targets['t_algo_x']))
    if 't_value_x' in targets:
        out['nu'] = float(np.log10(targets['t_value_x']))
    if 't_floor_x' in targets:
        out['g_C_inf'] = float(np.log10(targets['t_floor_x']))
    ref = replace(base, **out)
    if 't_bill_x' in targets:
        out['g_p'] = float(ref.g_C0 - np.log10(targets['t_bill_x']))
    if 't_lag_mo' in targets:
        lag_mo = float(targets['t_lag_mo'])
        lag_yr = lag_mo / 12.0
        speed = ref.g_C0 + ref.g_a
        out['Delta0'] = lag_yr * speed
        if merged:
            out['delta_dev'], out['delta_rel'] = split_delta(12.0 / lag_mo)
        else:
            out['delta_dev'], out['delta_rel'] = channels_from_lag(lag_yr, speed,
                                                                   ref.g_a_F + ref.g_CF0)
        ref = replace(ref, Delta0=out['Delta0'])
    if 't_profit_B' in targets:
        out['W0'] = float(targets['t_profit_B'] / _profit_scale(ref))
    return out


## Monte Carlo: joint draws from the calibration ranges

`PARAM_RANGES` encodes the per-parameter distributions from the calibration master (uniform;
triangular for $\theta$; lognormal for $\Delta_0$; $g^F_a$ drawn as a scale of $g_a$; $\eta$ from
the discrete menu). `sample_params` draws them **jointly** (independent per parameter) and
`monte_carlo` runs `simulate` per draw, returning headline-stat distributions, a subsample of
$\Pi_t$ paths for a fan chart, and an NPV-vs-$\tau$ distribution for question (b).

In [ ]:
#| export
# ---- Cell E8: parameter ranges, joint sampling, Monte Carlo ----
# Distributions per design section 5. Each entry: (kind, *args).
#   uniform:    ('uniform', lo, hi)
#   triangular: ('triangular', lo, mode, hi)
#   lognormal:  ('lognormal', mu, sigma)     [drawn as exp(Normal(mu, sigma))]
#   scale_of:   ('scale_of', other_name, lo, hi)   [param = draw(other) * U(lo,hi)]
#   choice:     ('choice', [values...])
PARAM_RANGES = {
    'g_C0':     ('uniform', 0.556, 0.724),
    'g_C_inf':  ('uniform', 0.05, 0.30),
    'xi':       ('uniform', 0.1, 1.0),
    'g_a':      ('uniform', 0.30, 0.544),
    'alpha':    ('uniform', 0.2, 0.8),
    'eta':      ('choice', [1.0, 0.61, 0.0, -2.0]),
    'rho0':     ('uniform', 0.1, 0.5),
    'gamma':    ('uniform', 0.0, 0.4),  # capped: gamma >~0.4 blows up inside horizon (N4)
    'g_a_F':    ('scale_of', 'g_a', 0.6, 0.8),
    'g_CF0':    ('uniform', 0.3, 0.7),
    'g_CF_inf': ('uniform', 0.05, 0.20),
    'xi_F':     ('uniform', 0.1, 1.0),
    'Delta0':   ('lognormal', np.log(0.7), 0.421),   # median 0.7, ~90% CI [0.35, 1.4]
    'split':    ('uniform', 0.3, 0.8),
    'delta_dev':('uniform', 0.08, 0.40),
    'delta_rel':('uniform', 0.12, 0.75),
    'W0':       ('uniform', 100.0, 1000.0),
    'nu':       ('uniform', 0.043, 0.478),  # D-039 log10 units (was [0.1, 1.1] natural)
    'x_mid':    ('uniform', 2.0, 20.0),
    'theta':    ('triangular', 0.15, 0.35, 0.6),
    'phi_RD':   ('uniform', 0.3, 2.0),
    'ell':      ('uniform', 0.25, 1.0),
    'S0':       ('uniform', 15.0, 100.0),
    'g_p':      ('uniform', 0.130, 0.282),  # D-039 log10 units; 1.35x/yr (hardware-only) .. ~1.9x/yr; central ~1.75x/yr reconciles g_C 4.2x/yr with cost growth 2.4x/yr
    'r':        ('uniform', 0.03, 0.15),
}

def _draw_one(kind_args, rng, drawn):
    kind = kind_args[0]
    if kind == 'uniform':
        return float(rng.uniform(kind_args[1], kind_args[2]))
    if kind == 'triangular':
        return float(rng.triangular(kind_args[1], kind_args[2], kind_args[3]))
    if kind == 'lognormal':
        return float(np.exp(rng.normal(kind_args[1], kind_args[2])))
    if kind == 'choice':
        return kind_args[1][int(rng.integers(len(kind_args[1])))]
    if kind == 'scale_of':
        base = drawn[kind_args[1]]
        return float(base * rng.uniform(kind_args[2], kind_args[3]))
    raise ValueError(kind)

def sample_params(rng, p_base):
    """Draw one Params jointly from PARAM_RANGES (independent per parameter, with the g_a_F scale
    coupling). Extension dials and scenario fields are inherited from p_base."""
    drawn = {}
    # resolve non-coupled first so scale_of can reference them
    order = [k for k in PARAM_RANGES if PARAM_RANGES[k][0] != 'scale_of'] + \
            [k for k in PARAM_RANGES if PARAM_RANGES[k][0] == 'scale_of']
    for k in order:
        drawn[k] = _draw_one(PARAM_RANGES[k], rng, drawn)
    return replace(p_base, **drawn)

def monte_carlo(n, p_base, seed=0, taus=(0.0, 0.5, 1.0), n_delay=60):
    """Joint-draw forecast (design section 3/5). Runs simulate per draw; returns headline-stat
    distributions plus a subsample (~50) of Pi_t paths for the fan chart. The NPV-vs-tau
    distribution for question (b) is computed on the first `n_delay` draws only (each extra tau is
    an extra simulate), which keeps the headline stats over all n draws affordable."""
    rng = np.random.default_rng(seed)
    npvs = np.empty(n); crossings = np.empty(n); within5 = np.empty(n, dtype=bool)
    stays = np.empty(n, dtype=bool); term_gap = np.empty(n); cum_profits = np.empty(n)
    blowup = np.empty(n, dtype=bool)   # leader path passes +25 OOM inside horizon (N4 singularity)
    tgrid = None
    paths = []
    n_paths = min(50, n)
    n_delay = min(n_delay, n)
    delay_npvs = np.full((n_delay, len(taus)), np.nan)
    for i in range(n):
        p = sample_params(rng, p_base)
        s = simulate(p)                                # base release rule (p_base.tau)
        h = headline(s, p)
        if tgrid is None:
            tgrid = s['t']
        npvs[i] = h['npv']; crossings[i] = h['sign_crossing_year']
        within5[i] = h['positive_within_5yr']; stays[i] = h['stays_positive']
        term_gap[i] = h['terminal_gap']; cum_profits[i] = float(s['cum_profit'][-1])
        blowup[i] = bool(np.nanmax(s['x_L']) > 25.0)
        if i < n_paths:
            paths.append(s['profit'])
        if i < n_delay:
            for j, tau in enumerate(taus):
                sj = s if tau == p.tau else simulate(replace(p, tau=tau))
                delay_npvs[i, j] = float(sj['npv_cum'][-1])
    return dict(
        t=tgrid, npvs=npvs, crossings=crossings, within5=within5, stays=stays,
        terminal_gap=term_gap, paths=np.array(paths), n=n, cum_profits=cum_profits,
        blowup=blowup, blowup_frac=float(blowup.mean()),
        p_npv_positive_sane=float((npvs[~blowup] > 0).mean()) if (~blowup).any() else float('nan'),
        p_profitable_within_5yr=float(within5.mean()),
        p_npv_positive=float((npvs > 0).mean()),
        p_profitable=float(np.isfinite(crossings).mean()),
        median_crossing=float(np.nanmedian(crossings)) if np.isfinite(crossings).any() else float('nan'),
        p_cumprofit_positive=float((cum_profits > 0).mean()),
        p_cumprofit_positive_sane=float((cum_profits[~blowup] > 0).mean()) if (~blowup).any() else float('nan'),
        taus=list(taus), delay_npvs=delay_npvs,
        delay_helps_frac=float((delay_npvs.argmax(axis=1) > 0).mean()),
    )


def _draw_dict(rng):
    """One joint draw from PARAM_RANGES as a plain dict (same logic as sample_params)."""
    drawn = {}
    order = [k for k in PARAM_RANGES if PARAM_RANGES[k][0] != 'scale_of'] + \
            [k for k in PARAM_RANGES if PARAM_RANGES[k][0] == 'scale_of']
    for k in order:
        drawn[k] = _draw_one(PARAM_RANGES[k], rng, drawn)
    return drawn


# merged catch-up delta (base-model levels 1-3). At these levels the follower has NO engine of its
# own (g_a_F, follower compute pinned to 0), so ALL of its motion is catch-up: the single effective
# rate delta must supply the leader's FULL speed. split_delta routes it entirely through delta_rel,
# acting on the full gap, so xdot^F = delta*(x^L - x^F) exactly.
_DELTA_DEV_DEFAULT = Params().delta_dev
_DELTA_ALGO_SHARE = 0.3
# merged-delta prior for Monte Carlo: lognormal, median 1.5/yr, ~90% CI [0.66, 3.4]/yr.
# pure-catch-up L1: delta*Delta balances the full leader speed 1.07/0.7 ~= 1.5; supersedes 0.36
# which assumed follower own growth (that returns at Level 4, where the ~0.25 wedge sets delta~0.36).
MERGED_DELTA_RANGE = ('lognormal', 0.405, 0.5)

def split_delta(delta_total):
    """Base-model mapping: the merged catch-up rate delta acts entirely on the capability gap,
    i.e. (delta_dev, delta_rel) = (0, delta), so catch-up = delta*(x^L - x^F) EXACTLY (the base
    model the widget's Level 1 presents). Level 3 unpacks delta into the two channels; the
    effective single rate then is delta_rel + algo_share*delta_dev (see D-034)."""
    return 0.0, float(delta_total)

def mc_draw_batch(p_base, n, seed=0, n_points=200, sample_keys=None, merge_delta=False,
                  target_ranges=None, param_ranges=None):
    """Per-draw records for the live Monte-Carlo view. Each record carries the sampled values,
    the (downsampled) trajectories the widget plots, and a few headline scalars.

    D-037: sampling is TARGET-SPACE wherever a target exists. `sample_keys` may mix
    TARGET_RANGES keys (drawn in natural units and inverted per draw via invert_targets;
    merge_delta selects the merged vs channels lag inversion) with PARAM_RANGES keys (free dials,
    drawn in parameter space). sample_keys=None -> every PARAM_RANGES key (legacy full param
    space, no targets). Everything not sampled stays pinned at p_base. The `params` record shows
    the drawn targets in their natural units (for the MC inspector strip). Trajectories are
    downsampled to <= n_points points; every draw shares the same time grid (T, dt from p_base)."""
    # target_ranges / param_ranges: optional overrides (e.g. user-narrowed sampling ranges,
    # layered over the module defaults by the widget); None -> the module-level dicts.
    TR = TARGET_RANGES if target_ranges is None else target_ranges
    PR = PARAM_RANGES if param_ranges is None else param_ranges
    rng = np.random.default_rng(seed)
    keys = list(PR) if sample_keys is None else list(sample_keys)
    tkeys = [k for k in keys if k in TR]
    raw = [k for k in keys if k in PR]
    raw_plain = [k for k in raw if PR[k][0] != 'scale_of']
    raw_scaled = [k for k in raw if PR[k][0] == 'scale_of']
    out = []
    idx = None
    for _ in range(n):
        drawn = {}
        for k in raw_plain:
            drawn[k] = _draw_one(PR[k], rng, drawn)
        targets = {tk: float(_draw_one(TR[tk], rng, {})) for tk in tkeys}
        # invert the non-lag/non-profit targets first, so scale_of raws (g_a_F ~ g_a) couple to the
        # DRAWN algo target; then draw those raws; then the lag/profit inversions (which need them).
        t1 = {k: v for k, v in targets.items() if k not in ('t_lag_mo', 't_profit_B')}
        inv = invert_targets(t1, replace(p_base, **drawn), merged=merge_delta)
        for k in raw_scaled:
            ctx = dict(drawn); ctx.update(inv); ctx.setdefault('g_a', p_base.g_a)
            drawn[k] = _draw_one(PR[k], rng, ctx)
        t2 = {k: v for k, v in targets.items() if k in ('t_lag_mo', 't_profit_B')}
        if t2:
            inv.update(invert_targets(t2, replace(p_base, **drawn, **inv), merged=merge_delta))
        p = replace(p_base, **drawn, **inv)
        s = simulate(p)
        if idx is None:
            nt = len(s['t']); k = max(1, int(np.ceil(nt / n_points)))
            idx = np.arange(0, nt, k)
            if idx[-1] != nt - 1:
                idx = np.append(idx, nt - 1)
        prof = s['profit']
        pos = prof >= 0.0
        crossing = float(s['t'][int(np.argmax(pos))]) if pos.any() else float('nan')
        out.append(dict(
            params={**drawn, **targets},
            t=s['t'][idx], profit=prof[idx],
            x_L=s['x_L'][idx], x_F=s['x_F'][idx], Delta=s['Delta'][idx],
            revenue=s['revenue'][idx], cost=s['cost'][idx], cum_profit=s['cum_profit'][idx],
            crossing=crossing, cum_profit_T=float(s['cum_profit'][-1]),
            blowup=bool(np.nanmax(s['x_L']) > 25.0),
        ))
    return out


In [ ]:
#| export
# ---- Cell E8b: calibration sources + tight default simulation ranges (D-042) ----
# CAL_SOURCES is the documented per-parameter source table. It drives BOTH the widget's
# calibration modals (source rows with [use] / [use as range]) AND the tight DEFAULT simulation
# ranges below. Keyed by PARAMETER name; `value` is in the TARGET's natural units where a target
# drives the parameter, in parameter units for free dials, an option string for eta. Faithful to
# Notes/calibration_master.md — no invented sources. `ci` = a source's own documented interval;
# ci_default=False marks a judgment band (kept for [use as range]) that does NOT enter the
# default simulation span.
_LAG_SOURCES = [
    dict(source="Epoch ECI benchmark lag", value=3.5, unit="mo",
         note="benchmark basis — a lower bound (benchmaxxing)"),
    dict(source="METR agentic evals", value=8.0, unit="mo", note="the central reading"),
    dict(source="Private benchmarks", value=9.0, unit="mo", note="~8–10-month readings"),
    dict(source="Cottier 2024a — compute-cost basis", value=15.0, unit="mo",
         note="stale compute basis; the upper reading"),
]
CAL_SOURCES = {
    "g_C0": [
        dict(source="Epoch (Sevilla & Roldán 2024) — post-2018 frontier trend", value=4.2,
             unit="×/yr", note="several Epoch series agree; documented 90% CI [3.6, 4.9]",
             ci=(3.6, 4.9)),
        dict(source="Epoch — long 2010–24 trend", value=5.3, unit="×/yr",
             note="upper reading; top of the sampled range"),
    ],
    "g_a": [
        dict(source="Ho et al. 2024 — pretraining efficiency", value=2.8, unit="×/yr",
             note="95% CI 4.5–14.3-month doubling time"),
        dict(source="Whitfill et al. 2025 — independent estimate", value=3.5, unit="×/yr",
             note="independent method, agrees in range"),
    ],
    "Delta0": _LAG_SOURCES, "delta_dev": _LAG_SOURCES, "delta_rel": _LAG_SOURCES,
    "delta_total": _LAG_SOURCES,
    "nu": [
        dict(source="Per-task token blow-up (cost-side convexity)", value=1.73, unit="×/OOM",
             note="chosen central of the (1, 3] range"),
        dict(source="Frontier cost-intensity — upper bound", value=3.0, unit="×/OOM",
             note="the ×3-per-OOM cap"),
    ],
    "g_p": [
        dict(source="Cottier et al. 2024b — training-bill growth", value=2.4, unit="×/yr",
             note="the baseline anchor; documented CI [2.0, 2.9]", ci=(2.0, 2.9)),
        dict(source="Hardware-only price-performance (Hobbhahn 2023)", value=3.11, unit="×/yr",
             note="implies prices fall only ×1.35/yr — the documented anchor tension"),
    ],
    "g_C_inf": [
        dict(source="Hobbhahn 2023 — hardware-only trend", value=1.35, unit="×/yr",
             note="floor LEVEL is our extrapolation; power binds ~2030 (Sevilla et al. 2024)"),
    ],
    "W0": [
        dict(source="Observed frontier gross profit — low reading", value=40.0, unit="\\$B/yr",
             note="OpenAI ~\\$20B ARR plus the rest of the frontier"),
        dict(source="Observed frontier gross profit — high reading", value=60.0, unit="\\$B/yr",
             note="Anthropic ~\\$60B run-rate at >65% margin, mid-2026"),
    ],
    "theta": [
        dict(source="Cournot floor, n = 5 leaders", value=0.17, unit="", note="ρ ≈ 1/(n+1)"),
        dict(source="Cournot floor, n = 3 leaders", value=0.25, unit="", note="ρ ≈ 1/(n+1)"),
        dict(source="Triangulated central (differentiation / lock-in)", value=0.35, unit="",
             note="documented central band 0.3–0.4; NOT the 60–80% gross margin (Lerner-index "
                  "trap)", ci=(0.30, 0.40), ci_default=False),
    ],
    "S0": [dict(source="2026 combined frontier training-compute spend", value=40.0,
                unit="\\$B/yr", note="anchors the cost scale")],
    "ell": [dict(source="Frontier run + integration time", value=0.5, unit="yr",
                 note="range 0.25–1 yr")],
    "phi_RD": [dict(source="R&D staff + experiments ≈ the final training run", value=1.0,
                    unit="×", note="≈ half of frontier-lab spend is R&D overhead")],
    "x_mid": [
        dict(source="Early-commoditization reference", value=2.0, unit="OOM",
             note="value saturates early"),
        dict(source="Mid reference", value=5.0, unit="OOM", note=""),
        dict(source="Harvest-continues reference", value=10.0, unit="OOM",
             note="the bend sits beyond the horizon"),
    ],
    "g_a_F": [
        dict(source="Scale-bias low, 0.6 × leader (Gundlach)", value=0.27, unit="OOM/yr", note=""),
        dict(source="Scale-bias central, 0.7 × leader (Gundlach)", value=0.31, unit="OOM/yr",
             note=""),
        dict(source="Scale-bias high, 0.8 × leader (Gundlach)", value=0.36, unit="OOM/yr",
             note="one source, three readings"),
    ],
    "eta": [
        dict(source="Whitfill & Wu 2025 — substitutes (σ = 2.58)", value="0.61", unit="",
             note="N = 27, 4 labs"),
        dict(source="Whitfill & Wu 2025 — complements (scale control)", value="-2 (complements)",
             unit="", note="sign flips on the K_train control"),
    ],
    "gamma": [dict(source="Tentative default (no observable yet)", value=0.2, unit="/OOM", note="")],
    "rho0": [dict(source="Tentative default (no observable yet)", value=0.3, unit="", note="")],
    "xi": [dict(source="Scenario default — sweep it", value=0.3, unit="/yr", note="")],
    "xi_F": [dict(source="Scenario default", value=0.3, unit="/yr", note="")],
    "split": [dict(source="Placeholder (open question)", value=0.5, unit="", note="")],
    "g_CF0": [dict(source="Follower build-out placeholder", value=0.5, unit="OOM/yr", note="")],
    "g_CF_inf": [dict(source="Follower floor placeholder", value=0.10, unit="OOM/yr", note="")],
    "alpha": [dict(source="Placeholder (no observable)", value=0.5, unit="", note="")],
    "r": [dict(source="Standard discount rate", value=0.08, unit="/yr",
               note="user-cost extension only")],
}

# Per-SOURCE reputation grades (D-043 chips; mockup layout_mockups.html): each source row
# gets its own letter — A solid measurement · B reasonable anchor · C judgment / weakly
# identified · F free choice or scenario default. Parameter-LEVEL grades stay app-side
# (ui/content.GRADES, from calibration_master); the lists follow each key's row order.
# The lag keys share ONE row list (_LAG_SOURCES), so grading Delta0 grades all four.
_SOURCE_GRADES = {
    'g_C0': ['A', 'B'], 'g_a': ['A', 'A'], 'Delta0': ['B', 'A', 'C', 'C'],
    'nu': ['C', 'C'], 'g_p': ['B', 'B'], 'g_C_inf': ['C'], 'W0': ['B', 'B'],
    'theta': ['C', 'C', 'C'], 'S0': ['C'], 'ell': ['B'], 'phi_RD': ['C'],
    'x_mid': ['C', 'C', 'C'], 'g_a_F': ['B', 'B', 'B'], 'eta': ['B', 'B'],
    'gamma': ['C'], 'rho0': ['C'], 'xi': ['F'], 'xi_F': ['F'], 'split': ['F'],
    'g_CF0': ['C'], 'g_CF_inf': ['C'], 'alpha': ['F'], 'r': ['C'],
}
for _k, _gs in _SOURCE_GRADES.items():
    for _row, _g in zip(CAL_SOURCES[_k], _gs):
        _row['grade'] = _g


def lognormal_from_ci(lo, hi):
    """Lognormal from its 90%-CI endpoints: geometric-mid mu, sigma = ln(hi/lo)/(2·1.645)."""
    return ('lognormal', float(np.log(np.sqrt(lo * hi))), float(np.log(hi / lo) / 3.29))


def dist_bounds(rng):
    """Natural-unit endpoints of a distribution: uniform/triangular bounds, lognormal ~90% CI."""
    if rng[0] == 'lognormal':
        med = float(np.exp(rng[1]))
        return med * float(np.exp(-1.645 * rng[2])), med * float(np.exp(1.645 * rng[2]))
    if rng[0] == 'triangular':
        return float(rng[1]), float(rng[3])
    return float(rng[1]), float(rng[2])


def source_span(pkey):
    """[min, max] across a parameter's documented source values and measured CIs (D-042)."""
    vals = []
    for rw in CAL_SOURCES.get(pkey, []):
        if isinstance(rw.get('value'), (int, float)):
            vals.append(float(rw['value']))
        ci = rw.get('ci')
        if ci is not None and rw.get('ci_default', True):
            vals += [float(ci[0]), float(ci[1])]
    return (min(vals), max(vals)) if vals else None


def _tight(key, pkey, kind='uniform'):
    """Tight default simulation range for one dimension: the source span, clipped to the
    envelope (TARGET_RANGES / PARAM_RANGES bounds)."""
    elo, ehi = dist_bounds(TARGET_RANGES.get(key) or PARAM_RANGES[key])
    slo, shi = source_span(pkey)
    lo, hi = max(float(slo), elo), min(float(shi), ehi)
    return lognormal_from_ci(lo, hi) if kind == 'lognormal' else ('uniform', lo, hi)


# D-042: the DEFAULT simulation range per MC dimension (target keys for targets, parameter keys
# for free dials), derived from CAL_SOURCES. Every dimension NOT listed here has no practical
# multi-source basis and defaults to a POINT: it starts in spot mode and is not sampled until
# the user widens its range (the envelope caps how far).
SIM_DEFAULT = {
    't_compute_x': _tight('t_compute_x', 'g_C0'),              # [3.6, 5.3] ×/yr (points + Epoch CI)
    't_algo_x':    _tight('t_algo_x', 'g_a'),                  # [2.8, 3.5] ×/yr
    't_lag_mo':    _tight('t_lag_mo', 'Delta0', 'lognormal'),  # lognormal, 90% CI [3.5, 15] mo —
                                                               # the lognormal shape fits the source
                                                               # span naturally (benchmaxxing skew)
    't_bill_x':    _tight('t_bill_x', 'g_p'),                  # [2.0, 3.11] ×/yr (points + Cottier CI)
    't_profit_B':  _tight('t_profit_B', 'W0'),                 # [40, 60] $B/yr
    't_value_x':   _tight('t_value_x', 'nu'),                  # [1.73, 3.0] ×/OOM
    'theta':       _tight('theta', 'theta'),                   # [0.17, 0.35]; the 0.3–0.4 judgment
                                                               # band is excluded (ci_default=False)
    'g_a_F':       ('scale_of', 'g_a', 0.6, 0.8),              # Gundlach's own documented 0.6–0.8×
                                                               # band (single source, three readings)
}


In [ ]:
print("Sampled parameters:", list(PARAM_RANGES.keys()))
rng = np.random.default_rng(0)
demo = sample_params(rng, Params())
print("one joint draw -> theta=%.3f  nu=%.3f  x_mid=%.2f  Delta0=%.3f  eta=%s"
      % (demo.theta, demo.nu, demo.x_mid, demo.Delta0, demo.eta))

## Full example simulation

A complete run at the bounded illustrative parameters ($\gamma=0$), with capability paths, the
gap, the revenue/cost decomposition, the profit flow, and cumulative NPV.

In [ ]:
from dataclasses import replace as _rep
pe = _rep(Params(), gamma=0.0)
se = simulate(pe); he = headline(se, pe)
print("HEADLINE (bounded example):")
for k, v in he.items(): print(f"  {k:22s} {v}")

fig, axs = plt.subplots(2, 2, figsize=(11.5, 7.2))
axs[0,0].plot(se['t'], se['x_L'], color=PAL["blue"], label="$x^L$")
axs[0,0].plot(se['t'], se['x_R'], color=PAL["aqua"], ls="--", label="$x^R$ (served)")
axs[0,0].plot(se['t'], se['x_F'], color=PAL["orange"], label="$x^F$")
style(axs[0,0], "Capability paths", "year", "OOM"); axs[0,0].legend(frameon=False, fontsize=9)
axs[0,1].plot(se['t'], se['revenue'], color=PAL["green"], label="revenue")
axs[0,1].plot(se['t'], se['cost'], color=PAL["critical"], label="cost")
style(axs[0,1], "Revenue vs cost", "year", "$/yr  ($B)"); axs[0,1].legend(frameon=False, fontsize=9)
axs[1,0].plot(se['t'], se['profit'], color=PAL["blue"])
axs[1,0].fill_between(se['t'], se['profit'], 0, where=se['profit']>=0, color=PAL["good"], alpha=0.18)
axs[1,0].fill_between(se['t'], se['profit'], 0, where=se['profit']<0, color=PAL["critical"], alpha=0.14)
style(axs[1,0], "Profit flow $\\Pi^L_t$", "year", "$/yr  ($B)")
axs[1,1].plot(se['t'], se['npv_cum'], color=PAL["violet"])
style(axs[1,1], "Cumulative NPV $\\int_0^t e^{-rs}\\Pi\\,ds$", "year", "$B")
fig.tight_layout(); plt.show()

## Release-delay comparison (question b)

$\Pi^L_t$ under release delays $\tau\in\{0,3,6,12,18,24\}$ months, and NPV as a function of $\tau$
with the arg-max marked. Withholding trades forgone current revenue against a slower follower.

In [ ]:
taus = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
dc = delay_comparison(pe, taus=taus)
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.0))
cols = [PAL["blue"], PAL["green"], PAL["magenta"], PAL["yellow"], PAL["orange"], PAL["violet"]]
for path, tau, c in zip(dc['paths'], taus, cols):
    axs[0].plot(dc['t'], path, color=c, label=f"$\\tau$={int(tau*12)}mo")
style(axs[0], "Profit flow by release delay", "year", "$/yr  ($B)")
axs[0].legend(frameon=False, fontsize=8, ncol=2)
axs[1].plot([t*12 for t in taus], dc['npvs'], color=PAL["blue"], marker="o", ms=5)
bi = int(np.argmax(dc['npvs']))
axs[1].scatter([taus[bi]*12], [dc['npvs'][bi]], color=PAL["good"], s=70, zorder=5, label="arg max")
style(axs[1], "NPV vs release delay", "release delay $\\tau$ (months)", "NPV  ($B)")
axs[1].legend(frameon=False, fontsize=9)
fig.tight_layout(); plt.show()
print("best tau (months):", dc['best_tau']*12, " best NPV:", round(dc['best_npv'],1))

## Monte Carlo demo (n = 200)

Joint draws from `PARAM_RANGES`: a fan chart of $\Pi_t$ paths, the NPV distribution, and the
headline probability the leader is profitable within 5 years. (Runs `simulate` 200+ times — takes
tens of seconds.)

In [ ]:
mc = monte_carlo(200, Params(), seed=1)
print(f"P(profitable within 5yr) = {mc['p_profitable_within_5yr']:.3f}")
print(f"P(NPV > 0)               = {mc['p_npv_positive']:.3f}")
print(f"fraction of draws where delaying helps NPV = {mc['delay_helps_frac']:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.0))
for path in mc['paths']:
    axs[0].plot(mc['t'], path, color=PAL["blue"], alpha=0.12, lw=1)
axs[0].plot(mc['t'], np.median(mc['paths'], axis=0), color=PAL["blue"], lw=2.4, label="median")
style(axs[0], "Profit fan chart (50 draws)", "year", "$/yr  ($B)")
axs[0].legend(frameon=False, fontsize=9)
finite = mc['npvs'][np.isfinite(mc['npvs'])]
clip = np.clip(finite, np.percentile(finite,2), np.percentile(finite,98))
axs[1].hist(clip, bins=30, color=PAL["violet"], alpha=0.85)
axs[1].axvline(0, color=PAL["critical"], lw=1.5)
style(axs[1], "NPV distribution (2-98 pct clip)", "NPV  ($B)", "count")
fig.tight_layout(); plt.show()

## Notes on provisional pieces

- **Explosive default ($\gamma=0.5$).** The provisional default sits in the super-exponential RSI
  regime (finite-time singularity ~8 yr); `psi_share` flags it at ~1.0. The intended
  harvest-vs-commoditization economics live in the bounded regime ($\gamma$ small or 0). This is a
  finding for the calibration session, not a settled default.
- **II.4 gap-dependent $\theta$** uses the provisional normalized-exponential form (D-027).
- **II.6 ownership / user-cost** applies a simplified Jorgenson user-cost factor
  $(r+\delta_K+g_p)/(r+g_p)$ (the depreciation load). It does **not** fully capture CAPEX
  front-loading in timing — flagged provisional; a faithful amortization is future work.
- Every `Params` default marked `provisional -> calibration session` is a placeholder the widget
  exposes as a control. The harvest condition (N5) and the acceptance tests live in
  `tests/test_model.py`.